# MovieLens 数据探索 (EDA)

对 MovieLens 数据集进行探索性数据分析，了解评分分布、用户活跃度、电影流行度、稀疏性等关键特征，为后续建模提供数据驱动的设计依据。

In [ ]:
from __future__ import annotations
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import seaborn as sns

matplotlib.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

from src.data.load_data import load_movielens, resolve_movielens_dir, make_sample_movielens
from src.config import RAW_DIR

%matplotlib inline

In [ ]:
USE_SAMPLE = True

if USE_SAMPLE:
    data = make_sample_movielens()
    print("使用样本数据进行测试")
else:
    data_dir = resolve_movielens_dir()
    data = load_movielens(data_dir)
    print(f"加载数据: {data_dir}")

ratings = data["ratings"]
movies = data["movies"]
tags = data["tags"]
print(f"Ratings: {ratings.shape}, Movies: {movies.shape}, Tags: {tags.shape}")

## 1. 数据集概览

In [ ]:
print("=== Ratings ===")
print(ratings.info())
print(f"\n评分范围: {ratings['rating'].min():.1f} - {ratings['rating'].max():.1f}")
print(f"用户数: {ratings['userId'].nunique():,}")
print(f"电影数: {ratings['movieId'].nunique():,}")
print(f"稀疏度: {1 - len(ratings) / (ratings['userId'].nunique() * ratings['movieId'].nunique()):.4%}")
print(f"\n=== Movies ===")
print(movies.info())
print(f"\n类型示例: {movies['genres'].iloc[0]}")
print(f"\n=== Tags ===")
print(tags.info())

## 2. 评分分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ratings["rating"].hist(bins=10, edgecolor="white", ax=ax1, color="steelblue")
ax1.set_title("评分分布直方图")
ax1.set_xlabel("Rating")
ax1.set_ylabel("Count")

ax2 = axes[1]
rating_counts = ratings["rating"].value_counts().sort_index()
ax2.bar(rating_counts.index.astype(str), rating_counts.values / rating_counts.sum() * 100, color="steelblue")
ax2.set_title("评分分布 (百分比)")
ax2.set_xlabel("Rating")
ax2.set_ylabel("%")

plt.tight_layout()
plt.show()

print(f"评分统计:")
print(f"  均值: {ratings['rating'].mean():.3f}")
print(f"  中位数: {ratings['rating'].median():.1f}")
print(f"  标准差: {ratings['rating'].std():.3f}")
print(f"  高分 (>=4.0) 占比: {(ratings['rating'] >= 4.0).mean():.2%}")

## 3. 电影类型分布

In [ ]:
genre_counts = movies["genres"].str.split("|").explode().value_counts()
genre_counts = genre_counts[genre_counts.index != "(no genres listed)"]

fig, ax = plt.subplots(figsize=(10, 6))
genre_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("电影类型分布")
ax.set_xlabel("电影数量")
plt.tight_layout()
plt.show()

genre_per_movie = movies["genres"].str.split("|").apply(len)
print(f"每部电影平均类型数: {genre_per_movie.mean():.2f}")
print(f"最多类型数: {genre_per_movie.max()}")

## 4. 用户活跃度分布

In [ ]:
user_activity = ratings.groupby("userId").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.hist(user_activity, bins=50, edgecolor="white", color="steelblue")
ax1.set_title("用户评分数分布")
ax1.set_xlabel("Ratings per user")
ax1.set_ylabel("User count")

ax2 = axes[1]
ax2.hist(np.log1p(user_activity), bins=50, edgecolor="white", color="coral")
ax2.set_title("用户评分数分布 (log1p)")
ax2.set_xlabel("log1p(Ratings per user)")
ax2.set_ylabel("User count")

plt.tight_layout()
plt.show()

print(f"用户评分数统计:")
print(f"  均值: {user_activity.mean():.1f}")
print(f"  中位数: {user_activity.median():.0f}")
print(f"  最小: {user_activity.min()}, 最大: {user_activity.max()}")
print(f"  仅评分 1 次的用户占比: {(user_activity == 1).mean():.2%}")

## 5. 电影流行度分布 (长尾效应)

In [ ]:
movie_popularity = ratings.groupby("movieId").agg(
    rating_count=("rating", "count"),
    rating_mean=("rating", "mean"),
).sort_values("rating_count", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.bar(range(len(movie_popularity)), movie_popularity["rating_count"], color="steelblue", width=1.0)
ax1.set_title("电影评分数排名 (Long Tail)")
ax1.set_xlabel("Movie rank")
ax1.set_ylabel("Rating count")
ax1.set_yscale("log")

ax2 = axes[2]
ax2.hist(movie_popularity["rating_count"], bins=50, edgecolor="white", color="steelblue")
ax2.set_title("电影评分数分布")
ax2.set_xlabel("Rating count per movie")
ax2.set_ylabel("Movie count")

plt.tight_layout()
plt.show()

top_pct = 0.2
top_movies = movie_popularity.head(int(len(movie_popularity) * top_pct))
total_ratings = movie_popularity["rating_count"].sum()
print(f"前 {top_pct:.0%} 的电影占据了 {top_movies['rating_count'].sum() / total_ratings:.2%} 的评分")
print(f"最热门电影评分数: {movie_popularity['rating_count'].iloc[0]}")
print(f"仅 1 次评分的电影占比: {(movie_popularity['rating_count'] == 1).mean():.2%}")

## 6. 时间模式

In [ ]:
if "timestamp" in ratings.columns:
    ratings["dt"] = pd.to_datetime(ratings["timestamp"], unit="s")
    ratings["year_month"] = ratings["dt"].dt.to_period("M")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    monthly = ratings.groupby("year_month").size()
    ax1 = axes[0]
    monthly.plot(ax=ax1, color="steelblue")
    ax1.set_title("每月评分数量")
    ax1.set_ylabel("Rating count")
    ax1.tick_params(axis="x", rotation=45)

    ratings["hour"] = ratings["dt"].dt.hour
    hourly = ratings.groupby("hour").size()
    ax2 = axes[1]
    hourly.plot(kind="bar", ax=ax2, color="steelblue")
    ax2.set_title("每小时评分数量")
    ax2.set_xlabel("Hour of day")

    plt.tight_layout()
    plt.show()
else:
    print("样本数据无 timestamp 列，跳过时间分析")

## 7. 标签分析

In [ ]:
if not tags.empty and "tag" in tags.columns:
    tag_freq = tags["tag"].str.lower().str.strip().value_counts().head(20)

    fig, ax = plt.subplots(figsize=(10, 6))
    tag_freq.sort_values().plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("Top 20 最常见标签")
    ax.set_xlabel("出现次数")
    plt.tight_layout()
    plt.show()

    tags_per_movie = tags.groupby("movieId").size()
    print(f"每部电影平均标签数: {tags_per_movie.mean():.2f}")
else:
    print("无标签数据或样本数据中标签为空")

## 8. 稀疏矩阵可视化

In [ ]:
matrix = ratings.pivot_table(index="userId", columns="movieId", values="rating", fill_value=0.0)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(matrix.values[:50, :50], aspect="auto", cmap="YlOrRd", vmin=0, vmax=5)
ax.set_title(f"User-Movie Rating Matrix (first 50x50)\
Sparsity: {1 - (matrix.values > 0).mean():.2%}")
ax.set_xlabel("Movie index")
ax.set_ylabel("User index")
plt.colorbar(im, ax=ax, label="Rating")
plt.tight_layout()
plt.show()

n_users, n_movies = matrix.shape
n_ratings = (matrix.values > 0).sum()
print(f"User-Movie 矩阵: {n_users} × {n_movies}")
print(f"非零元素: {n_ratings:,} / {n_users * n_movies:,} = {n_ratings / (n_users * n_movies):.4%}")

## 9. 关键发现与设计决策

1. **评分分布偏差**: 大多数评分集中在 3-4 分，高分偏多——这是典型的正向偏差，需要在流行度计算中使用贝叶斯平滑。
2. **长尾分布**: 少数电影获得大量评分，大多数电影评分很少——需要最小评分数过滤。
3. **用户活跃度差异大**: 少数活跃用户贡献了大部分评分——冷启动问题不可忽视。
4. **稀疏性极高**: 典型的推荐场景，需要降维或相似度计算来处理。
5. **类型是重要内容信号**: Drama 和 Comedy 占比最高，类型特征将是内容滤波的重要输入。
6. **标签提供额外语义**: 用户标签可以捕捉类型之外的内容信息，适合 TF-IDF 特征化。